# AIC 2026 — OCR keyframe bằng **PaddleOCR-VL-1.6** trên Kaggle

Model: [`PaddlePaddle/PaddleOCR-VL-1.6`](https://huggingface.co/PaddlePaddle/PaddleOCR-VL-1.6) — VLM 0.9B (NaViT visual encoder + ERNIE-4.5), SOTA 96.33% trên OmniDocBench v1.6.

**Khác gì notebook cũ?** PaddleOCR-VL làm *cả detection lẫn recognition trong một lượt forward*, nên notebook này **bỏ hẳn PaddleOCR-det + VietOCR**. Không cần cài `paddlepaddle` → tránh được toàn bộ rắc rối CUDA/paddle trên Kaggle.

**Hai cách chạy model này:**

| Cách | Lệnh | Ghi chú |
|---|---|---|
| `transformers` (dùng ở notebook này) | `pip install "transformers>=5.0.0"` | Gọi thẳng model, nhẹ, kiểm soát được batch. Phù hợp keyframe (ảnh scene-text). |
| Pipeline `paddleocr[doc-parser]` | `pip install "paddleocr[doc-parser]>=3.6.0"` + `paddlepaddle-gpu==3.2.1` | Có thêm layout analysis, xuất Markdown. Nặng và thừa cho keyframe video. |
| vLLM server | `paddleocr genai_server --backend vllm` | Nhanh nhất nhưng **không cài chung env với `transformers>=5`** (xung đột version) và khó dựng trong 1 kernel Kaggle. |

⚠️ **Settings của Kaggle notebook (panel bên phải):**
- **Accelerator** → `GPU T4 x2` (hoặc P100)
- **Internet** → `ON` (bắt buộc: cần tải weight từ HuggingFace)


In [ ]:
# ============================================================
# CÀI ĐẶT — chạy ô này ĐẦU TIÊN trong kernel còn "sạch"
# ============================================================
import sys, subprocess

def sh(cmd):
    print('>>', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

PIP = f'{sys.executable} -m pip install -q'

# PaddleOCR-VL-1.6 yêu cầu transformers >= 5.0.0 (Kaggle mặc định còn 4.x)
sh(f'{PIP} -U "transformers>=5.0.0" "accelerate>=1.0" "huggingface_hub>=0.30"')

# KHÔNG cài vllm vào cùng env: version transformers mà vLLM cần xung đột với transformers>=5.

import transformers, torch
print('transformers :', transformers.__version__)
print('torch        :', torch.__version__, '| cuda', torch.version.cuda)
print('GPU          :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ')

# Nếu ô này báo lỗi import transformers (do bản cũ đã nạp sẵn),
# vào menu Run -> Restart & Clear Cell Outputs rồi chạy lại từ đầu.


## Cấu hình Dataset & tham số chạy

In [ ]:
import os
from pathlib import Path

# Tat tokenizer song song: notebook nay chay nhieu thread (1 thread / GPU),
# de bat se sinh canh bao va co the deadlock.
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# 1. Dataset mount ở /kaggle/input/<dataset-slug>  (KHÔNG có phần "datasets/<username>/")
#    Chạy  !ls /kaggle/input  nếu không chắc tên.
DATASET_DIRECTORY = Path('/kaggle/input/aic-dataset')

# 2. Thư mục output
DEMO_OUTPUT = Path('/kaggle/working/ocr_vl')

# 3. Folder keyframe chạy cho session này
TARGET_FOLDER = 'Keyframes_L22'

# 4. Tác vụ của PaddleOCR-VL. Các giá trị hợp lệ theo model card:
#      'ocr'      -> trả về text thuần (dùng cho keyframe: nhanh nhất, hợp retrieval)
#      'spotting' -> trả về text KÈM toạ độ box (chậm hơn, ảnh nhỏ sẽ được upscale x2)
#      'table' / 'formula' / 'chart' / 'seal' -> tác vụ chuyên biệt
TASK = 'ocr'

MODEL_PATH = 'PaddlePaddle/PaddleOCR-VL-1.6'
MODEL_ID = f'PaddleOCR-VL-1.6 ({TASK})'

# BATCH_SIZE tính TRÊN MỖI GPU. Model 0.9B fp16 chỉ ~2 GB weight, T4 có ~15 GB
# nên phần lớn VRAM dành cho activation + KV cache -> 8 là mức an toàn cho task 'ocr'.
# Giảm về 4 nếu dùng TASK='spotting' (ảnh bị upscale x2 nên tốn gấp ~4 lần).
BATCH_SIZE = 8 if TASK != 'spotting' else 4
MAX_NEW_TOKENS = 512    # keyframe ít chữ -> 256 là đủ và nhanh gần gấp đôi
CHECKPOINT_EVERY = 200  # ghi kết quả tạm mỗi N ảnh (phòng session Kaggle bị ngắt)
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}

DEMO_OUTPUT.mkdir(parents=True, exist_ok=True)
if not DATASET_DIRECTORY.is_dir():
    mounted = sorted(p.name for p in Path('/kaggle/input').iterdir()) if Path('/kaggle/input').is_dir() else []
    raise AssertionError(f'Khong thay dataset: {DATASET_DIRECTORY}. /kaggle/input dang co: {mounted}')
target_path = DATASET_DIRECTORY / TARGET_FOLDER
assert target_path.is_dir(), (f'Khong thay thu muc con {TARGET_FOLDER}. '
                              f'Dataset dang co: {sorted(p.name for p in DATASET_DIRECTORY.iterdir())[:20]}')

keyframe_roots = [target_path]
print(f'Session nay chay folder: {TARGET_FOLDER} | task = {TASK} | batch/GPU = {BATCH_SIZE}')


## Liệt kê toàn bộ keyframe trong target folder

In [ ]:
import re

VIDEO_ID_PATTERN = re.compile(r'^L\d{2}_V\d{3}$')

def find_video_dirs(root):
    base = root / 'keyframes'
    if not base.is_dir():
        base = root
    return [p for p in sorted(base.iterdir())
            if p.is_dir() and VIDEO_ID_PATTERN.match(p.name)]

def list_images(video_dir):
    return sorted((p for p in video_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS),
                  key=lambda p: (not p.stem.isdigit(), int(p.stem) if p.stem.isdigit() else 0, p.name))

all_video_dirs = [v for root in keyframe_roots for v in find_video_dirs(root)]
all_video_dirs.sort(key=lambda p: p.name)
assert all_video_dirs, 'Khong tim thay video nao'

samples = []  # [(video_dir, [image_path, ...])]
for video_dir in all_video_dirs:
    images = list_images(video_dir)
    if images:
        samples.append((video_dir, images))

total_keyframes = sum(len(imgs) for _, imgs in samples)
print(f'{len(samples)} video | tong {total_keyframes} keyframe')
for video_dir, images in samples[:20]:
    print(f'- {video_dir.name}: {len(images)} keyframe')
if len(samples) > 20:
    print(f'... va {len(samples) - 20} video nua')


## Tải PaddleOCR-VL-1.6 — một bản sao trên MỖI GPU

Kaggle cấp **2 × T4, mỗi con ~15 GB**. Model chỉ 0.9B (~2 GB ở fp16) nên **không shard model qua 2 GPU** — `device_map="auto"` sẽ cắt model làm đôi và phải đẩy activation qua PCIe mỗi token, chậm hơn chạy một GPU. Cách đúng là **data parallel**: nạp một bản sao đầy đủ lên `cuda:0` và `cuda:1`, rồi chia keyframe cho hai bên → throughput gần gấp đôi.

Weight chỉ tải về từ HuggingFace một lần (~2 GB), bản sao thứ hai đọc từ cache nên nhanh.

**Lưu ý dtype:** model card ghi bf16, nhưng **T4 và P100 không hỗ trợ bfloat16** (compute capability < 8.0). Ô dưới tự chọn `float16` trên các GPU đó. Tương tự `flash_attention_2` cần Ampere trở lên nên ở đây dùng `sdpa`.

In [ ]:
import time, torch
from transformers import AutoProcessor, AutoModelForImageTextToText

N_GPU = torch.cuda.device_count()
DEVICES = [f'cuda:{i}' for i in range(N_GPU)] if N_GPU else ['cpu']
for i in range(N_GPU):
    free, total = torch.cuda.mem_get_info(i)
    print(f'cuda:{i} = {torch.cuda.get_device_name(i)} | {total/2**30:.1f} GB')
print(f'-> se nap {len(DEVICES)} ban sao model:', DEVICES)

def pick_dtype(device):
    if device == 'cpu':
        return torch.float32
    # bfloat16 chi co tu Ampere (sm_80). T4 = sm_75, P100 = sm_60 -> phai dung float16.
    return torch.bfloat16 if torch.cuda.get_device_capability(device)[0] >= 8 else torch.float16

def pick_attn(device):
    if device == 'cpu':
        return 'sdpa'
    return 'flash_attention_2' if torch.cuda.get_device_capability(device)[0] >= 8 else 'sdpa'

def load_replica(device):
    """Nap 1 ban sao model + processor rieng cho mot GPU."""
    dtype, attn = pick_dtype(device), pick_attn(device)

    # Moi replica giu processor rieng: tokenizer/image_processor khong dam bao
    # thread-safe khi hai thread goi cung luc.
    proc = AutoProcessor.from_pretrained(MODEL_PATH)
    if getattr(proc, 'tokenizer', None) is not None:
        proc.tokenizer.padding_side = 'left'   # generate theo batch phai pad ben TRAI

    model, last = None, None
    for kw in ({'dtype': dtype, 'attn_implementation': attn},          # transformers v5
               {'torch_dtype': dtype, 'attn_implementation': attn},    # v4
               {'torch_dtype': dtype}):
        try:
            model = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, **kw)
            break
        except (TypeError, ValueError) as e:
            last = e
    if model is None:
        raise last

    model = model.to(device).eval()
    return {'device': device, 'model': model, 'processor': proc, 'dtype': dtype, 'attn': attn}

t0 = time.perf_counter()
REPLICAS = [load_replica(d) for d in DEVICES]
load_seconds = time.perf_counter() - t0

DTYPE = REPLICAS[0]['dtype']
n_params = sum(p.numel() for p in REPLICAS[0]['model'].parameters()) / 1e9
print(f'\nLoad {len(REPLICAS)} replica trong {load_seconds:.1f}s | {n_params:.2f}B params '
      f'| dtype={DTYPE} attn={REPLICAS[0]["attn"]}')
for i in range(N_GPU):
    print(f'  cuda:{i} da dung {torch.cuda.memory_allocated(i)/2**30:.2f} GB VRAM')


## Hàm OCR (chạy theo batch)

Prompt lấy đúng theo model card: `OCR:` / `Table Recognition:` / `Formula Recognition:` / `Chart Recognition:` / `Spotting:` / `Seal Recognition:`.

In [ ]:
import re
from PIL import Image

PROMPTS = {
    'ocr': 'OCR:',
    'table': 'Table Recognition:',
    'formula': 'Formula Recognition:',
    'chart': 'Chart Recognition:',
    'spotting': 'Spotting:',
    'seal': 'Seal Recognition:',
}
assert TASK in PROMPTS, f'TASK khong hop le: {TASK}'

SPOTTING_UPSCALE_THRESHOLD = 1500
MAX_PIXELS = 2048 * 28 * 28 if TASK == 'spotting' else 1280 * 28 * 28

# --- Doc min_pixels tu image processor ---
# Model card viet processor.image_processor.min_pixels, nhung transformers v5 da gop
# min_pixels/max_pixels vao image_processor.size = {'shortest_edge':..., 'longest_edge':...}
# nen attribute do khong con ton tai -> AttributeError. Doc theo nhieu duong cho chac.
def resolve_min_pixels(processor):
    ip = processor.image_processor
    v = getattr(ip, 'min_pixels', None)
    if isinstance(v, int) and v > 0:
        return v
    size = getattr(ip, 'size', None)
    if isinstance(size, dict):
        for key in ('shortest_edge', 'min_pixels'):
            if isinstance(size.get(key), int) and size[key] > 0:
                return size[key]
    return 144 * 28 * 28   # = 112896, dung gia tri trong preprocessor_config.json cua model

MIN_PIXELS = resolve_min_pixels(REPLICAS[0]['processor'])
IMAGES_KWARGS = {'size': {'shortest_edge': MIN_PIXELS, 'longest_edge': MAX_PIXELS}}
print(f'image_processor.size = {getattr(REPLICAS[0]["processor"].image_processor, "size", None)}')
print(f'-> dung min_pixels={MIN_PIXELS}, max_pixels={MAX_PIXELS}')

def load_image(path):
    """Doc anh, upscale x2 cho task spotting neu anh nho (theo model card)."""
    image = Image.open(path).convert('RGB')
    image.load()
    if TASK == 'spotting':
        w, h = image.size
        if w < SPOTTING_UPSCALE_THRESHOLD and h < SPOTTING_UPSCALE_THRESHOLD:
            try:
                resample = Image.Resampling.LANCZOS
            except AttributeError:
                resample = Image.LANCZOS
            image = image.resize((w * 2, h * 2), resample)
    return image

_imgkw_warned = False

def _build_inputs(rep, images):
    global _imgkw_warned
    processor = rep['processor']
    convs = [[{'role': 'user', 'content': [{'type': 'image', 'image': im},
                                           {'type': 'text', 'text': PROMPTS[TASK]}]}]
             for im in images]
    payload = convs if len(convs) > 1 else convs[0]
    base = dict(add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt')
    if len(convs) > 1:
        base['padding'] = True
    try:
        out = processor.apply_chat_template(payload, images_kwargs=IMAGES_KWARGS, **base)
    except (TypeError, KeyError, ValueError) as e:
        # Bo qua images_kwargs neu processor khong nhan -> dung size mac dinh cua model.
        if not _imgkw_warned:
            print('!! images_kwargs bi tu choi, dung size mac dinh:', type(e).__name__, e)
            _imgkw_warned = True
        out = processor.apply_chat_template(payload, **base)
    return out.to(rep['model'].device)

@torch.inference_mode()
def ocr_images(rep, images):
    """Chay 1 batch anh tren MOT replica -> list chuoi ket qua tho."""
    inputs = _build_inputs(rep, images)
    prompt_len = inputs['input_ids'].shape[-1]
    out = rep['model'].generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    return [t.strip() for t in rep['processor'].batch_decode(out[:, prompt_len:],
                                                             skip_special_tokens=True)]

BOX_TAG = re.compile(r'<[^<>]{0,80}>')          # the kieu <box>, <ref>, ...
COORD_RUN = re.compile(r'\(?\[?\s*\d+\s*,\s*\d+\s*(,\s*\d+\s*,\s*\d+\s*)?\]?\)?')

def to_plain_text(raw):
    """Voi task 'spotting' output co lan toa do -> boc ra phan chu."""
    if TASK != 'spotting':
        return raw
    cleaned = COORD_RUN.sub(' ', BOX_TAG.sub(' ', raw))
    return re.sub(r'\s+', ' ', cleaned).strip()

def ocr_paths(rep, paths):
    """Chay 1 batch duong dan tren MOT replica -> list dict ket qua, co do thoi gian."""
    t = time.perf_counter()
    images = [load_image(p) for p in paths]
    io_s = time.perf_counter() - t

    t = time.perf_counter()
    try:
        raws = ocr_images(rep, images)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print(f'  !! OOM tren {rep["device"]} -> lui ve xu ly tung anh (giam BATCH_SIZE di)')
        raws = [ocr_images(rep, [im])[0] for im in images]
    infer_s = time.perf_counter() - t

    per_image = (io_s + infer_s) / max(len(paths), 1)
    return [{'io': io_s / len(paths), 'infer': infer_s / len(paths), 'total': per_image,
             'device': rep['device'], 'raw': r, 'text': to_plain_text(r)} for r in raws]


## Smoke test — chạy thử vài ảnh trước khi đốt vài giờ GPU

In [ ]:
probe_paths = [p for _, imgs in samples[:2] for p in imgs[:2]][:4]

# Thu tren TUNG replica de chac ca 2 GPU deu chay duoc (bat loi som truoc khi chay full).
for rep in REPLICAS:
    t0 = time.perf_counter()
    probe = ocr_paths(rep, probe_paths)
    print(f'===== {rep["device"]} | {(time.perf_counter()-t0)/len(probe_paths)*1000:.0f} ms/anh =====')
    for path, res in zip(probe_paths, probe):
        print(f'--- {path.name} ---')
        print(res['text'][:400] if res['text'] else '(khong co chu)')
    print()

if all(not r['text'] for r in probe):
    print('!! Tat ca deu rong. Kiem tra: anh co chu that khong, TASK co dung khong, '
          'dtype co bi bf16 tren T4 khong.')


## Xem tận mắt OCR có tốt không

Vẽ ảnh gốc cạnh text mà model đọc ra, để tự đánh giá trước khi chạy cả dataset.

- `TASK='ocr'`: chỉ có text, panel bên phải hiển thị nguyên văn.
- `TASK='spotting'`: model trả kèm toạ độ → vẽ luôn box lên ảnh. Phần bóc toạ độ là **best-effort** (format output của spotting không có trong model card), nên chuỗi `raw` luôn được in ra để bạn đối chiếu.

Chỉnh `VIZ_N` để xem nhiều/ít ảnh hơn.

In [ ]:
import textwrap
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

VIZ_N = 6   # so anh muon xem

# Lay VIZ_N anh rai deu ca folder thay vi 6 anh lien tiep cua cung 1 video
flat = [p for _, imgs in samples for p in imgs]
step = max(1, len(flat) // VIZ_N)
viz_paths = flat[::step][:VIZ_N]

# Chay qua replica dau tien, chia thanh cac batch BATCH_SIZE
viz_results = []
for i in range(0, len(viz_paths), BATCH_SIZE):
    viz_results.extend(ocr_paths(REPLICAS[0], viz_paths[i:i + BATCH_SIZE]))

# --- Boc toa do tu output cua task 'spotting' (best-effort) ---
QUAD = re.compile(r'(\d{1,5})\D{1,3}(\d{1,5})\D{1,3}(\d{1,5})\D{1,3}(\d{1,5})')

def parse_boxes(raw, width, height):
    """Tra ve [(x1,y1,x2,y2,text), ...]. Rong neu khong doc duoc toa do nao."""
    hits = list(QUAD.finditer(raw))
    if not hits:
        return []
    items = []
    for i, m in enumerate(hits):
        x1, y1, x2, y2 = (int(g) for g in m.groups())
        end = hits[i + 1].start() if i + 1 < len(hits) else len(raw)
        label = re.sub(r'\s+', ' ', BOX_TAG.sub(' ', raw[m.end():end])).strip()
        items.append([x1, y1, x2, y2, label])
    # Toa do thuong duoc chuan hoa ve 0..1000 (kieu ho Qwen) -> tu doan va scale lai
    biggest = max(max(it[:4]) for it in items)
    sx, sy = (width / 1000.0, height / 1000.0) if biggest <= 1000 else (1.0, 1.0)
    return [(x1 * sx, y1 * sy, x2 * sx, y2 * sy, t) for x1, y1, x2, y2, t in items]

rows = len(viz_paths)
fig, axes = plt.subplots(rows, 2, figsize=(15, 4.2 * rows),
                         gridspec_kw={'width_ratios': [3, 2]})
if rows == 1:
    axes = axes.reshape(1, 2)

for ax_img, ax_txt, path, res in zip(axes[:, 0], axes[:, 1], viz_paths, viz_results):
    image = Image.open(path).convert('RGB')
    ax_img.imshow(image)
    ax_img.set_axis_off()

    boxes = parse_boxes(res['raw'], *image.size) if TASK == 'spotting' else []
    for x1, y1, x2, y2, label in boxes:
        ax_img.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                            fill=False, linewidth=1.6, edgecolor='#ff2d55'))
        if label:
            ax_img.text(x1, max(y1 - 4, 8), label[:28], color='white', fontsize=7,
                        bbox=dict(facecolor='#ff2d55', edgecolor='none', pad=1))

    n_note = f' | {len(boxes)} box' if TASK == 'spotting' else ''
    ax_img.set_title(f'{path.parent.name}/{path.name}  ({res["total"]*1000:.0f} ms{n_note})',
                     fontsize=10)

    ax_txt.set_axis_off()
    body = res['text'] if res['text'] else '(model khong doc ra chu nao)'
    ax_txt.text(0, 1, textwrap.fill(body, width=52)[:1800],
                va='top', ha='left', fontsize=9, family='DejaVu Sans',
                bbox=dict(facecolor='#f5f5f5', edgecolor='#cccccc', pad=8))

plt.tight_layout()
plt.show()

# In raw de doi chieu, nhat la khi dung TASK='spotting'
print('=' * 70)
for path, res in zip(viz_paths, viz_results):
    print(f'--- {path.parent.name}/{path.name} | {len(res["raw"])} ky tu raw ---')
    print(res['raw'][:600])
    print()


## Chạy toàn bộ dataset trên **cả 2 GPU** (có checkpoint & resume)

Mỗi GPU chạy trong một thread riêng. Thread Python không phải nút thắt ở đây vì `model.generate()` giải phóng GIL trong lúc chờ CUDA, nên hai GPU thực sự chạy song song.

Số replica đúng bằng số worker, và mỗi replica được lấy ra từ một hàng đợi — nên **không bao giờ có hai batch cùng chạy trên một GPU** (nếu không sẽ OOM).

Kernel Kaggle bị kill sau 12h. Ô này ghi checkpoint mỗi `CHECKPOINT_EVERY` ảnh; nếu session đứt, chỉ cần **chạy lại ô này** — nó tự bỏ qua các keyframe đã xong.

In [ ]:
import json, queue
from concurrent.futures import ThreadPoolExecutor

records = []
ckpt_path = DEMO_OUTPUT / f'{TARGET_FOLDER}_ckpt.jsonl'

# --- resume: nap lai nhung gi da lam ---
done = set()
if ckpt_path.exists():
    with ckpt_path.open(encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            r = json.loads(line)
            records.append(r)
            done.add((r['video_id'], r['keyframe']))
    print(f'Resume: da co {len(done)} keyframe trong checkpoint, se bo qua.')

todo = [(vd, p) for vd, imgs in samples for p in imgs if (vd.name, p.name) not in done]
batches = [todo[i:i + BATCH_SIZE] for i in range(0, len(todo), BATCH_SIZE)]
print(f'Con lai {len(todo)} / {total_keyframes} keyframe -> {len(batches)} batch '
      f'chia deu cho {len(REPLICAS)} GPU.')

# Hang doi replica: worker muon chay phai muon 1 replica va tra lai sau khi xong.
# So worker == so replica nen moi GPU luon chi phuc vu dung 1 batch tai mot thoi diem.
replica_pool = queue.Queue()
for rep in REPLICAS:
    replica_pool.put(rep)

def run_batch(batch):
    rep = replica_pool.get()
    try:
        if rep['device'] != 'cpu':
            torch.cuda.set_device(rep['device'])
        return ocr_paths(rep, [p for _, p in batch])
    finally:
        replica_pool.put(rep)

run_started = time.perf_counter()
processed = 0
per_device = {rep['device']: 0 for rep in REPLICAS}
ckpt_file = ckpt_path.open('a', encoding='utf-8')
try:
    with ThreadPoolExecutor(max_workers=len(REPLICAS)) as ex:
        # Nop het batch mot luot; ThreadPoolExecutor tu dieu phoi. Doc ket qua theo
        # dung thu tu nop de checkpoint luon on dinh, GPU van chay song song.
        futures = [ex.submit(run_batch, b) for b in batches]

        for batch, fut in zip(batches, futures):
            results = fut.result()
            for (video_dir, image_path), res in zip(batch, results):
                rec = {'video_id': video_dir.name, 'keyframe': image_path.name,
                       'ms': round(res['total'] * 1000), 'device': res['device'],
                       'text': res['text'], 'raw': res['raw']}
                records.append(rec)
                ckpt_file.write(json.dumps(rec, ensure_ascii=False) + '\n')
                per_device[res['device']] += 1

            processed += len(batch)
            if processed % CHECKPOINT_EVERY < BATCH_SIZE or processed >= len(todo):
                ckpt_file.flush()
                elapsed = time.perf_counter() - run_started
                rate = processed / max(elapsed, 1e-9)
                eta = (len(todo) - processed) / max(rate, 1e-9)
                split = ' '.join(f'{d.replace("cuda:", "gpu")}={n}' for d, n in per_device.items())
                print(f'[{processed}/{len(todo)}] {rate:.2f} anh/s | ETA {eta/60:.1f} phut '
                      f'| {split} | {records[-1]["video_id"]}/{records[-1]["keyframe"]} '
                      f'-> {records[-1]["text"][:50]}')
finally:
    ckpt_file.close()

wall_seconds = time.perf_counter() - run_started
print(f'\nXong {len(todo)} anh trong {wall_seconds/60:.1f} phut '
      f'({len(todo)/max(wall_seconds,1e-9):.2f} anh/s tren {len(REPLICAS)} GPU). '
      f'Tong ban ghi: {len(records)}')
print('Phan chia cong viec:', per_device)


## Xuất báo cáo JSON

In [ ]:
import statistics

mean_ms = statistics.fmean([r['ms'] for r in records]) if records else 0
with_text = [r for r in records if r['text']]

report = {
    'model': MODEL_ID,
    'model_path': MODEL_PATH,
    'task': TASK,
    'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    'n_replicas': len(REPLICAS),
    'dtype': str(DTYPE),
    'batch_size_per_gpu': BATCH_SIZE,
    'max_new_tokens': MAX_NEW_TOKENS,
    'load_seconds': round(load_seconds, 1),
    'ms_per_keyframe': {'mean': round(mean_ms, 1)},
    'wall_seconds': round(wall_seconds, 1),
    'throughput_img_per_s': round(len(records) / max(wall_seconds, 1e-9), 3),
    'total_keyframes_processed': len(records),
    'keyframes_with_text': len(with_text),
    'results': [{'video_id': r['video_id'], 'keyframe': r['keyframe'],
                 'ms': r['ms'], 'text': r['text']} for r in records],
}
path = DEMO_OUTPUT / f'{TARGET_FOLDER}_paddleocrvl_report.json'
path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')

print('Da luu:', path)
print(f'{len(with_text)}/{len(records)} keyframe co chu | trung binh {mean_ms:.0f} ms/anh/GPU '
      f'| throughput {report["throughput_img_per_s"]} anh/s tren {len(REPLICAS)} GPU')


In [ ]:
from IPython.display import FileLink
FileLink(str(path.relative_to('/kaggle/working')))


## Phụ lục — nếu muốn dùng pipeline `paddleocr[doc-parser]` thay vì `transformers`

Route này cho thêm layout analysis + xuất Markdown, nhưng cần cài `paddlepaddle` (nặng, dễ lệch CUDA trên Kaggle). **Không chạy chung kernel với các ô ở trên** — hãy tách sang notebook riêng.

```python
!python -m pip install paddlepaddle-gpu==3.2.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!python -m pip install -U "paddleocr[doc-parser]>=3.6.0"

from paddleocr import PaddleOCRVL
pipeline = PaddleOCRVL(pipeline_version="v1.6")
for res in pipeline.predict("/kaggle/input/aic-dataset/Keyframes_L22/L22_V001/001.jpg"):
    res.print()
    res.save_to_json(save_path="/kaggle/working/out")
    res.save_to_markdown(save_path="/kaggle/working/out")
```
